In [1]:
import os
import warnings

if 'Modeling' in os.path.abspath("").split('/'):
    os.chdir('..')
if 'Notebooks' in os.path.abspath("").split('/'):
    os.chdir('..')

project_root = os.path.abspath("")

warnings.filterwarnings('ignore')

In [2]:
import numpy as np

In [3]:
loaded = np.load('./Generated/Spectrums/exec_morlets.npz')

In [4]:
results_arr = []
i = 0
while f'power_{i}' in loaded:
    power = loaded[f'power_{i}']
    phase = loaded[f'phase_{i}']
    s_id = int(loaded[f'subject_id_{i}'])
    t_id = int(loaded[f'trial_id_{i}'])
    gender = str(loaded[f'gender_{i}'])
    handiness = str(loaded[f'handiness_{i}'])
    age = int(loaded[f'age_{i}'])
    label = int(loaded[f'label_{i}'])
    img = loaded[f'img_{i}']
    task_type = str(loaded[f'task_type_{i}'])
    
    results_arr.append([power, phase, s_id, t_id, gender, handiness, age, label, img, task_type])
    i += 1

power, phase, s_id, t_id, gender, handiness, age, label, img, task_type = results_arr[0]

In [5]:
import psutil
import os

process = psutil.Process(os.getpid())
print(f"Используется памяти: {process.memory_info().rss / 1024 ** 2:.2f} MB")

Используется памяти: 37941.20 MB


In [6]:
len(results_arr)

1260

In [7]:
subject_id_set  = set()
trial_id_set    = set()
gender_set      = set()
handiness_set   = set()
age_set         = set()
labels_set      = set()
task_type_set   = set()

for power, phase, s_id, t_id, gender, handiness, age, label, img, task_type in results_arr:
    labels_set.add(label)
    task_type_set.add(task_type)
    subject_id_set.add(s_id)
    trial_id_set.add(t_id)
    gender_set.add(gender)
    handiness_set.add(handiness)  # исправлено имя переменной, чтобы не перезаписывать множество
    age_set.add(age)

print("Labels:", labels_set)
print("Task types:", task_type_set)
print("Subject IDs:", subject_id_set)
print("Trial IDs:", trial_id_set)
print("Genders:", gender_set)
print("Handiness:", handiness_set)
print("Ages:", age_set)

Labels: {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, -1}
Task types: {'g', 'r'}
Subject IDs: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34}
Trial IDs: {1, 2}
Genders: {'m', 'n', 'f'}
Handiness: {'l', 'r', 'n'}
Ages: {32, 41, 18, 19, 20, 21, 22, 23, 30}


## Create dir structure

In [8]:
def mkdir_p(path):
    try:
        os.mkdir(path)
    except FileExistsError:
        pass
    except PermissionError:
        print(f"Permission denied: Unable to create '{path}'.")
    except Exception as e:
        print(f"An error occurred: {e}")

In [12]:
from tqdm import tqdm

In [13]:
last_s_id = None
last_t_id = None
current_trial_path = None
morlets_path = f'{project_root}/Generated/Spectrums/exec_morlets'
mkdir_p(morlets_path)
save_dict = {}
sample_id = 0
for power, phase, s_id, t_id, gender, handiness, age, label, img, task_type in tqdm(results_arr):

    if s_id != last_s_id or t_id != last_t_id:        
        last_s_id = s_id
        last_t_id = t_id
        sample_id = 0

    sample_id += 1
    save_dict = {}

    save_dict[f'power'] = power                             # (ch, freq, time)
    save_dict[f'phase'] = phase                             # (ch, freq, time)
    save_dict[f'subject_id'] = np.array(s_id)
    save_dict[f'trial_id'] = np.array(t_id)
    save_dict[f'gender'] = np.array(gender, dtype='U1')
    save_dict[f'handiness']  = np.array(handiness, dtype='U1')
    save_dict[f'age'] = np.array(age, dtype=int)
    save_dict[f'label']  = np.array(label, dtype=int)
    save_dict[f'img'] = np.array(img, dtype=int)
    save_dict[f'task_type'] = np.array(task_type, dtype='U1')

    current_subject_path = f'{morlets_path}/S_{s_id}'
    mkdir_p(current_subject_path)
    current_trial_path = f'{current_subject_path}/Trial_{last_t_id}'
    mkdir_p(current_trial_path)

    np.savez(f'{current_trial_path}/exec_morlets_{last_s_id}_{last_t_id}_{sample_id}.npz', **save_dict)


  0%|          | 3/1260 [00:00<00:50, 24.91it/s]

100%|██████████| 1260/1260 [00:52<00:00, 24.19it/s]
